In [1]:
# Depemdecy for the docx document conversion 
!pip install docx2txt

# Ingestion Pipline Implemenation 
## Load the data and convert those into the LangChain Documents 

### When we have to load the Microsoft Excel Sheet Data, that case over best apporach should be following :
- Directly converting Excel to CSV and using a dedicated CSV processor (like CSVLoader) is the best practice.
- Step of this process

```
1. We need to load excel file in Pandas DataFrame.
2. We need to preprocess this dataframe like replacing empty cell as empty string instead of NaN also strip strings inside the each cell in dataframe.
3. Load this modified Pandas DataFrame into the dedicated csv folder.
4. With the help of the LangChain CSVLoader Convert this csv data into the LandChain Documents.
```

- But here is also the fast but less reliable alternate way to directly convert Excel File into the LangChain Documents.
```python
Code : 

from langchain_community.document_loaders import UnstructuredExcelLoader

loader = UnstructuredExcelLoader("data_source.xlsx", mode="elements")
docs = loader.load()
```

- Comparison Table for the best understand diffrence between CSVLoader vs UnstructuredExcelLoader
 
| Feature | Excel to CSV + `CSVLoader` (Industry Standard) | `UnstructuredExcelLoader` (Prototype Only) |
| :--- | :--- | :--- |
| **Speed & Memory** | **Extremely Fast.** Low RAM consumption. | **Slow.** Resource-heavy because it parses fonts, styles, and layouts. |
| **Data Control** | **High.** You strip out hidden rows, empty columns, and broken formulas first. | **Low.** It extracts text blindly, often mixing up columns and rows. |
| **Token Efficiency** | **Excellent.** Structured text maximizes LLM embedding accuracy. | **Poor.** Adds massive layout garbage text, wasting your LLM token budget. |
| **Scalability** | Handles millions of rows easily via streaming. | Crashes or times out on large files. |


## 1. Loading the Documents 

In [2]:
import os 

from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader, TextLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
import pandas as pd

In [3]:
class DataLoaderManager:

    def __init__(self):
        
        self.pdf_folder_path = "data/pdfs_data"
        self.docs_folder_path = "data/docs_data"
        self.text_folder_path = "data/text_data"
        self.sheet_folder_path = "data/sheet_data"
        self.csv_folder_path = "data/csv_data"

    def load_pdfs(self):

        # Extract the all files name first 
        all_files = os.listdir(self.pdf_folder_path)

        # define list to store this all contents inside it 
        all_documents = []
        num_docs = 0

        for filename in all_files:
            
            if filename.lower().strip().endswith(".pdf"):

                # Extract the full file path 
                file_path = os.path.join(self.pdf_folder_path, filename)

                # Load the pdf file content 
                pdf_loader = PyMuPDFLoader(file_path)
                doc = pdf_loader.load()

                # add metadata part 
                for d in doc:
                    d.metadata["source_type"] = "pdf"

                # here every pdf containes the diffrent no of pages so that why we use extend() instead of the append()
                all_documents.extend(doc)
                num_docs += 1

        # Print the log message 
        print("\nPDFs => Extracted Documents Size : ", len(all_documents))
        print("PDFs => No of Documents Extracted : ", num_docs)

        return all_documents

        
    def load_docs(self):

        # Extract the all files name first 
        all_files = os.listdir(self.docs_folder_path)

        # define list to store this all contents inside it 
        all_documents = []
        num_docs = 0

        for filename in all_files:
            
            if filename.lower().strip().endswith(".docx"):

                # Extract the full file path 
                file_path = os.path.join(self.docs_folder_path, filename)

                # Load the pdf file content 
                doc_loader = Docx2txtLoader(file_path)
                doc = doc_loader.load()

                # add metadata part 
                for d in doc:
                    d.metadata["source_type"] = "docx"

                # here every pdf containes the diffrent no of pages so that why we use extend() instead of the append()
                all_documents.extend(doc)
                num_docs += 1

        # Print the log message 
        print("\nDocuments => Extracted Documents Size : ", len(all_documents))
        print("Documents => No of Documents Extracted : ", num_docs)

        return all_documents

    def __convert_excel_to_clean_csv(self, excel_file_path, csv_file_path):

        # first load hte excel file 
        # here sheet_name = 0 means we want to access the very first sheet inside the MS Workbook (every excel file can containes multiple sheets and we can see this in bottom tab section
        # We want to fill empty cell as empty string becoz if we not do explicitly then pandas add NaN which further causes ERROR in ingestion Pipline
        df = pd.read_excel(excel_file_path, sheet_name = 0).fillna("")

        # this map fnx we apply on entire dataset so it apply each cell of row and work on row by row (top modt left cell to bottom most right cell)
        # if any cell is string and containes some leading and traling space that case we remove it to improve ingestion pipline
        df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

        # save this modified sheet in given folder as csv file 
        df.to_csv(csv_file_path, index = False)     

    def load_sheets(self):

        # Extract the all files name first 
        all_files = os.listdir(self.sheet_folder_path)

        # define list to store this all contents inside it 
        all_documents = []
        num_docs = 0

        for filename in all_files:

            if filename.lower().strip().endswith(".xlsx"):

                # Extract the full file path 
                file_path = os.path.join(self.sheet_folder_path, filename)

                # We convert this sheet into the csv and store this csv in desire folder 
                self.__convert_excel_to_clean_csv(file_path, os.path.join(self.csv_folder_path, f"{filename.replace('.xlsx', '')}.csv"))

                # Load the csv file content 
                csv_filename = filename.replace(".xlsx", ".csv")
                csv_path = os.path.join(self.csv_folder_path, csv_filename)
                csv_loader = CSVLoader(csv_path)

                doc = csv_loader.load()

                # add metadata part 
                for d in doc:
                    d.metadata["source_type"] = "sheet"

                all_documents.extend(doc)
                num_docs += 1

        # Print the log message 
        print("\nSheets => Extracted Documents Size : ", len(all_documents))
        print("Sheets => No of Documents Extracted : ", num_docs)

        return all_documents
        

    def load_texts(self):
        
        # Extract the all files name first 
        all_files = os.listdir(self.text_folder_path)

        # define list to store this all contents inside it 
        all_documents = []
        num_docs = 0

        for filename in all_files:
            
            if filename.lower().strip().endswith(".txt"):

                # Extract the full file path 
                file_path = os.path.join(self.text_folder_path, filename)

                # Load the pdf file content 
                text_loader = TextLoader(file_path)
                doc = text_loader.load()

                # add metadata part 
                for d in doc:
                    d.metadata["source_type"] = "text"

                # here every pdf containes the diffrent no of pages so that why we use extend() instead of the append()
                all_documents.extend(doc)
                num_docs += 1

        # Print the log message 
        print("\nText File => Extracted Documents Size : ", len(all_documents))
        print("Text File => No of Documents Extracted : ", num_docs)

        return all_documents

    def load_all_documents(self):
        
        pdf_documents = self.load_pdfs()
        docs_documents = self.load_docs()
        texts_documents = self.load_texts()
        sheets_documents = self.load_sheets()

        return {
            "pdf_documents": pdf_documents, 
            "docs_documents": docs_documents,
            "texts_documents": texts_documents,
            "sheets_documents": sheets_documents,
            "all_documents": pdf_documents + docs_documents + texts_documents + sheets_documents
        }

In [4]:
data_loader_manager = DataLoaderManager()
all_documents = data_loader_manager.load_all_documents()


PDFs => Extracted Documents Size :  16
PDFs => No of Documents Extracted :  3

Documents => Extracted Documents Size :  3
Documents => No of Documents Extracted :  3

Text File => Extracted Documents Size :  3
Text File => No of Documents Extracted :  3

Sheets => Extracted Documents Size :  33
Sheets => No of Documents Extracted :  3


In [5]:
all_docs = all_documents["all_documents"]

print(len(all_docs))

# Note : for the sheet data every row is consider as documents not whole document
print(all_docs[34])

55
page_content='Employee ID: EMP106
Employee Name: Sonal Parmar
Department: Kitchen
Role: Sweet Preparation
Shift Timing: 7 AM - 3 PM
Experience: 4 Years
Employment Type: Full-Time
Village: Madhavpur
Monthly Salary (INR): 23000' metadata={'source': 'data/csv_data\\employee_salary_and_departments.csv', 'row': 5, 'source_type': 'sheet'}


## Sheet/Excel Data Handling Strategies in RAG Systems

Structured data such as Excel sheets and CSV files are handled differently from PDFs and text documents in RAG systems because tables already contain organized semantic information.

Different projects and industries use different ingestion strategies depending on the use case.

---

### 1. Row-Wise Document Strategy (Most Common)

## Concept

Each row of the table is converted into an individual `Document` object.

Example:

| Product | Price | Stock |
|---|---|---|
| Mango Pickle | 180 | Available |

becomes:

```python
Document(
    page_content="Product: Mango Pickle, Price: 180, Stock: Available"
)
```

---

#### Common Workflow

```text
Excel File
↓
Convert to CSV
↓
CSVLoader
↓
Each row becomes one Document
```

---

#### Common Modules and Tools

### Excel Reading
```python
pandas.read_excel()
```

### CSV Conversion
```python
DataFrame.to_csv()
```

### LangChain Loader
```python
CSVLoader
```

---

#### Advantages

- Simple architecture
- Easy debugging
- Better semantic precision
- Good retrieval quality
- Natural row-level meaning
- Easy metadata management

---

#### Best Use Cases

- Product catalogs
- Inventory systems
- Pricing tables
- Employee records
- Order tracking systems
- Customer databases

---

#### Recommendation for Learning Phase

✅ BEST choice for beginner and intermediate RAG systems.

This is the strategy currently used in the project.

---

### 2. Whole-Sheet Document Strategy

#### Concept

The entire Excel sheet is converted into one large `Document`.

Example:

```python
Document(
    page_content="Entire table content..."
)
```

---

#### Common Workflow

```text
Excel File
↓
Read Entire Sheet
↓
Convert whole table into text
↓
Single Document Creation
```

---

#### Common Modules and Tools

### Excel Reading
```python
pandas.read_excel()
```

### Text Conversion
```python
DataFrame.to_string()
```

or custom formatting logic.

---

#### Advantages

- Preserves complete table context
- Useful when rows are strongly connected
- Better for small datasets

---

#### Disadvantages

- Large document size
- Poor retrieval precision
- Harder semantic matching
- Chunking becomes difficult

---

#### Best Use Cases

- Financial reports
- Analytics summaries
- Small structured reports
- Research tables

---

#### Recommendation

⚠️ Not ideal for beginner RAG systems with large tables.

---

### 3. Hybrid Strategy (Advanced Industry Approach)

#### Concept

Professional RAG systems sometimes combine:

1. Row-wise documents
2. Table summaries

Together.

---

#### Workflow

```text
Excel File
↓
Row-wise Documents
↓
Table Summary Generation
↓
Store Both in Vector Database
```

---

#### Example

#### Row-Level Document

```text
Product: Mango Pickle
Price: 180
Stock: Available
```

#### Table Summary

```text
This table contains pricing and stock information for VillageTaste Foods products.
```

---

#### Common Modules and Tools

#### Row Processing
```python
CSVLoader
```

#### Summary Generation
- manual summaries
- LLM-generated summaries
- pandas aggregation logic

---

#### Advantages

- Best retrieval flexibility
- Supports both detailed and global questions
- Better enterprise-level retrieval quality

---

#### Disadvantages

- More complex architecture
- Requires additional processing logic
- Higher storage usage

---

#### Best Use Cases

- Enterprise RAG systems
- Analytics platforms
- Large business databases
- Advanced AI assistants

---

#### Important Chunking Insight

Structured table rows are already small semantic units.

Example:

```text
Product: Mango Pickle
Price: 180
Stock: Available
```

This already contains complete meaning.

Therefore:

> Aggressive chunking of row-wise table data is usually NOT recommended.

Improper chunking may:
- break relationships
- reduce semantic quality
- hurt retrieval accuracy

---

#### Best Choice for Current Project

For the current VillageTaste Foods chatbot project:

> Row-wise document strategy is the best choice.

Reason:
- product data
- employee records
- order tracking
- pricing information

all naturally fit row-level semantic retrieval.

## 2. Chuncking Documents

### For the dirrent document we apply diffrent splitting / chuncking technique : 
| Data Type | Recommended Chunking |
|---|---|
| PDFs | RecursiveCharacterTextSplitter |
| TXT | RecursiveCharacterTextSplitter |
| DOCX | RecursiveCharacterTextSplitter with separators |
| Sheets | Minimal or no chunking |

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
class DataChunckingManager:

    def __init__(self):

        # here we define over splitters 

        # PDF and Text Document Chunck splitter 
        self.pdf_and_text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = 500,
            chunk_overlap = 50
        )

        # for the document spllitter 
        # we give priority of splitting with the help of the seperators
        '''
        it first try to split paragraph if not possible then 
        try to split with lines and so one in given constaints as chunk_size and chunk_overlap 
        ''' 
        
        self.docs_splitter = RecursiveCharacterTextSplitter(
            chunk_size = 500,
            chunk_overlap = 50,
            separators = [
                "\n\n",
                "\n",
                ". ",
                " ", 
                ""
            ]
        )

    def chunck_pdfs_and_texts_files(self, pdfs_documents, texts_documents):
        documents = pdfs_documents + texts_documents

        chunck_docs = self.pdf_and_text_splitter.split_documents(documents)

        # define some log messages 
        print("\nPDFs and Text Document chunking : Length of chunck = ", len(chunck_docs))

        return chunck_docs

    def chunck_docx_files(self, docs_documents):

        chunck_docs = self.docs_splitter.split_documents(docs_documents)

        # define some log messages 
        print("\nMSWord Document chunking : Length of chunck = ", len(chunck_docs))

        return chunck_docs
        
    def chunck_sheet_files(self, sheet_documents):
        
        print("\nMSExcel Document chunking : Length of chunck = ", len(sheet_documents))
        
        return sheet_documents 

    def get_chunck_data(self, pdfs_documents, texts_documents, docs_documents, sheet_documents):
        
        pdf_text_chuncks = self.chunck_pdfs_and_texts_files(pdfs_documents, texts_documents)
        docs_chuncks = self.chunck_docx_files(docs_documents)
        sheet_chuncks = self.chunck_sheet_files(sheet_documents)

        all_chunck = pdf_text_chuncks + docs_chuncks + sheet_chuncks

        print("\nAll Documents chunck size : ", len(all_chunck))

        return all_chunck 

In [8]:
chunck_manager = DataChunckingManager()
chuncks_documents = chunck_manager.get_chunck_data(all_documents["pdf_documents"], all_documents["texts_documents"], all_documents["docs_documents"], all_documents["sheets_documents"])


PDFs and Text Document chunking : Length of chunck =  63

MSWord Document chunking : Length of chunck =  36

MSExcel Document chunking : Length of chunck =  33

All Documents chunck size :  132


In [9]:
chuncks_documents[0]

Document(metadata={'producer': 'Skia/PDF m149 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': 'data/pdfs_data\\company_overview.pdf', 'file_path': 'data/pdfs_data\\company_overview.pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'company_overview', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_type': 'pdf'}, page_content='VillageTaste Foods – Company Overview \nIntroduction \nVillageTaste Foods is an Indian village-based food business focused on delivering traditional \nhomemade food products prepared using authentic local recipes. The business was started by a \nsmall family group with the goal of preserving village-style food culture and providing healthier \nalternatives to heavily processed packaged foods. \nThe company combines traditional cooking methods with modern packaging and delivery')

### Some Detailed about over ```MSWord Document Chuncking``` 
#### DOCX (Word Document) Chunking Strategies in RAG Systems

DOCX files usually contain:
- paragraphs
- headings
- sections
- formatted text

Therefore, chunking strategies for DOCX files are slightly different from plain text files.

---

### 1. RecursiveCharacterTextSplitter (Most Common)

#### Concept

The most widely used approach for DOCX chunking is:

```python
RecursiveCharacterTextSplitter
```

This splitter tries to preserve text structure by splitting recursively using separators.

---

#### Recommended Configuration

```python
RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=60,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
```

---

#### How It Works

The splitter tries:
1. paragraph split
2. line split
3. sentence split
4. word split
5. character split

This helps preserve paragraph-level meaning.

---

#### Advantages

- Simple and flexible
- Works well for most DOCX files
- Production-friendly
- Easy to implement

---

#### Best Choice for Current Project

> Recommended for the current VillageTaste Foods chatbot project.

---

### 2. Header/Markdown-Based Chunking (Advanced)

#### Concept

Some systems convert DOCX files into Markdown format and then apply:
- heading-aware chunking
- section-aware splitting

Example tools:
- MarkdownHeaderTextSplitter

---

#### Best Use Cases

- technical manuals
- structured reports
- research documents
- documentation systems

---

#### Advantages

- preserves section hierarchy
- better contextual grouping

---

#### Disadvantages

- more preprocessing required
- more complex pipeline

---

### 3. Semantic Chunking (Advanced Industry Method)

#### Concept

Semantic chunking uses:
- embeddings
- meaning similarity
- LLM reasoning

to determine chunk boundaries instead of fixed characters.

---

#### Advantages

- high semantic quality
- better retrieval performance

---

#### Disadvantages

- computationally expensive
- complex implementation
- harder for beginner projects

## 3. Embeddings Conversion 

In [10]:
from sentence_transformers import SentenceTransformer

In [11]:
class EmbeddingManager:

    def __init__(self, model_name = "all-MiniLM-L6-v2"):

        print("Model is Loading...")        
        self.model_name = model_name

        self.model = SentenceTransformer(self.model_name)
        print("Model Name : ", self.model_name)

    def generate_embeddings(self, texts):
        
        embeddings = self.model.encode(texts, show_progress_bar = True)

        # Add some logs indicators 
        print("Type of Embeddings : ", type(embeddings))
        print("Type of Embeddings Data : ", type(embeddings[0]))
        print("Embedding Dimensions : ", embeddings.shape)
        
        return embeddings 

In [12]:
embedding_manager = EmbeddingManager()

# first extract only page_contents inside the chunck_documents 
texts = [doc.page_content for doc in chuncks_documents]

embeddings = embedding_manager.generate_embeddings(texts)

Model is Loading...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model Name :  all-MiniLM-L6-v2


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Type of Embeddings :  <class 'numpy.ndarray'>
Type of Embeddings Data :  <class 'numpy.ndarray'>
Embedding Dimensions :  (132, 384)


## 4. Store Embeddings inside the VectorDB 

In [13]:
import chromadb
import uuid

In [14]:
class VectorStoreManager:

    def __init__(self, persist_directory = "data/vector_store", collection_name = "knowledge_base"):

        self.persist_directory  = persist_directory
        self.collection_name = collection_name

        self.client = None 
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):

        # Create directory where over database actually store data
        os.makedirs(self.persist_directory, exist_ok = True)

        # Create client / Handler for handle the database all things
        self.client = chromadb.PersistentClient(
            path = self.persist_directory
        )

        # Create location where we actually store over knowledge_base data
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description": "Knowledge_Base Creation to store pdfs, docx, sheets and text documents data"}
        )

    def add_documents(self, documents, embeddings):

        # Check first edge case 
        if len(documents) != len(embeddings):
            raise ValueError("No of Documents and No Document related Embeddings are not same")

        # Initialize some data structure for storing data
        ids = []
        all_metadatas = []
        all_documents_contents = []
        all_embeddings = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):

            # Store each documents unique id 
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            # Store each documents metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["doc_id"] = doc_id

            metadata["content_length"] = len(doc.page_content)
            all_metadatas.append(metadata)

            # Store each documents actual content 
            all_documents_contents.append(doc.page_content)

            # Store each documents related embeddings
            '''
            here we convert embeddings into the list becoz individual chunck related embeddings in form of numpy array 
            and overall chuncks embeddings stores inside the list 
            '''
            
            all_embeddings.append(emb.tolist())

        # Store this four things in VectorDB 
        self.collection.add(
            ids = ids,
            metadatas = all_metadatas,
            documents = all_documents_contents,
            embeddings = all_embeddings
        )

        # add some log for the clarification 
        print("Length of Database's Collection : ", self.collection.count())
        print("Total Documents Count : ", len(all_documents_contents))

In [15]:
vector_store_manager = VectorStoreManager()
vector_store_manager.add_documents(chuncks_documents, embeddings)

Length of Database's Collection :  528
Total Documents Count :  132


# Retriever Pipline Implemenation 

## Understanding ChromaDB `collection.query()` Return Structure

### Query Example

```python
results = self.vector_store.collection.query(
    query_embeddings=[query_embeddings.tolist()],
    n_results=top_k
)
```

This performs:

```text
Semantic similarity search inside the vector database.
```

The vector database compares:
- query embedding
with
- stored document chunk embeddings

and returns the most relevant chunks.

---

## What Does `collection.query()` Return?

ChromaDB returns a dictionary.

Example structure:

```python
results = {

    "ids": [...],

    "documents": [...],

    "metadatas": [...],

    "distances": [...],

    "embeddings": [...]   # optional
}
```

---

## Why Everything is Inside Lists?

ChromaDB supports multiple query embeddings at once.

Example:

```python
query_embeddings = [
    embedding1,
    embedding2
]
```

Therefore ChromaDB groups results query-wise.

---

## Example Structure

```python
results["documents"] = [

    [doc1, doc2, doc3],   # results for query 1

    [doc4, doc5, doc6]    # results for query 2
]
```

Outer list:
```text
represents each query
```

Inner list:
```text
represents retrieved documents for that query
```

---

## Why We Use `[0]`

In the project:

```python
query_embeddings=[query_embeddings.tolist()]
```

Only ONE query embedding is passed.

Therefore:

```python
results["documents"][0]
```

means:

```text
retrieved documents for the first query
```

Same logic applies to:

```python
results["ids"][0]
results["metadatas"][0]
results["distances"][0]
```

---

## Meaning of Each Returned Field

---

## 1. `results["documents"]`

Contains:
```text
actual retrieved chunk texts
```

Example:

```python
[
   "VillageTaste Foods sells pickles...",
   "Our snacks include..."
]
```

---

## 2. `results["ids"]`

Contains:
```text
unique IDs of retrieved chunks
```

Example:

```python
[
   "doc_123",
   "doc_456"
]
```

Useful for:
- tracking
- debugging
- updating/deleting chunks

---

## 3. `results["metadatas"]`

Contains:
```text
metadata related to retrieved chunks
```

Example:

```python
[
   {
      "source": "menu.pdf",
      "page": 2,
      "source_type": "pdf"
   }
]
```

Useful for:
- citations
- filtering
- source tracking
- debugging

---

## 4. `results["distances"]`

Contains:
```text
vector distance scores
```

Smaller distance:
```text
more semantically similar
```

Larger distance:
```text
less semantically similar
```

---

## Why We Convert Distance into Similarity Score?

Code:

```python
similarity_score = 1 - distance
```

Reason:

```text
distance is less intuitive for humans
```

Similarity score becomes easier to understand.

Higher similarity score:
```text
better semantic match
```

---

## Understanding This Condition

```python
if results["documents"] and results["documents"][0]:
```

---

### First Check

```python
results["documents"]
```

Checks whether:
```text
documents key contains data
```

---

### Second Check

```python
results["documents"][0]
```

Checks whether:
```text
first query actually retrieved documents
```

This is defensive programming to avoid empty retrieval errors.

---

## Complete Retrieval Flow

```text
User Query
↓
Query Embedding
↓
ChromaDB Similarity Search
↓
results dictionary returned
↓
Extract:
    ids
    documents
    metadatas
    distances
↓
Build ranked retrieval response
```

---

## Important Final Understanding

Vector databases do NOT only store vectors.

They typically store:
- embeddings
- actual document chunks
- metadata
- unique IDs

This makes RAG systems:
- searchable
- traceable
- explainable
- filterable

In [16]:
class RetreiverManager:

    def __init__(self, embedding_manager, vector_store):
        
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retreive(self, query, top_k = 5, score_threshold = 0.2):

        # Query => Embeddings 
        # Here embedding_manager expect the multiple sentences as queries but we have only one query 
        # so to access the given query embeddings from nested of list given by embedding_manager that we use 0th index 
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # retreive the top_k result 
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # Validate the retreived results
        # Here we check documents key containes valid data and inside this valid data we check it empty or not
        retreived_documents = []
        
        if results["documents"] and results["documents"][0]:

            # Extract data inside the results
            ids = results["ids"][0]
            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]

            for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):

                # Compute the cosine similarity 
                similarity_score = 1 - distance

                # Check retreived document compatible with score_threshold 
                if similarity_score >= score_threshold:

                    retreived_documents.append({
                        "doc_id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
            print(f"Retreived {len(retreived_documents)} Documents") 
                    
        else:
            print("No Related Document Retreived")

        return retreived_documents  

In [17]:
retreiver_manager = RetreiverManager(embedding_manager, vector_store_manager)

In [18]:
retreiver_manager.retreive("What is your VillageTaste Food offer food ?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Type of Embeddings :  <class 'numpy.ndarray'>
Type of Embeddings Data :  <class 'numpy.ndarray'>
Embedding Dimensions :  (1, 384)
Retreived 5 Documents


[{'doc_id': 'doc_be65f842-58a8-4a27-a6d6-e1989cdfe1c2',
  'document': 'VillageTaste Foods – Traditional Food Catalog \nAbout Us \nVillageTaste Foods is a small Indian village-based food business that specializes in preparing \nhomemade traditional foods using authentic recipes passed down through generations. Our \nproducts are prepared using fresh ingredients, traditional spices, and village-style cooking \nmethods. We focus on maintaining natural taste, hygiene, and quality in every product.',
  'metadata': {'format': 'PDF 1.4',
   'source': 'data/pdfs_data\\product_catalog.pdf',
   'keywords': '',
   'moddate': '',
   'content_length': 422,
   'creationdate': '',
   'doc_id': 'doc_be65f842-58a8-4a27-a6d6-e1989cdfe1c2',
   'page': 0,
   'title': 'product_catalog',
   'modDate': '',
   'creator': '',
   'creationDate': '',
   'total_pages': 7,
   'producer': 'Skia/PDF m149 Google Docs Renderer',
   'doc_index': 24,
   'subject': '',
   'file_path': 'data/pdfs_data\\product_catalog.pdf

# Main RAG Pipline Implementation
## Pipline : 
```
User Query
↓
Query Rewriter LLM
↓
Improved Search Query
↓
Retriever
↓
Better Retrieval
```

In [19]:
# !pip install langchain-google-genai

In [20]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

```
""" You rewrite search queries for a VillageTaste Food business chatbot. The knowledge base contains: - food products - company services - employee policies - order information Rewrite the user query to improve semantic retrieval. Keep the rewritten query concise. Return ONLY the rewritten query. Do not explain anything. User Query: {query} """
```

In [51]:
class RAGPipelineManager:

    def __init__(self, retreiver_manager):

        # Load the api key 
        load_dotenv()
        self.API_KEY = os.getenv("GEMINI_API_KEY")

        # Define query_refine_model
        self.query_refine_llm = ChatGoogleGenerativeAI(
            model = "gemini-2.5-flash",
            temperature = 0.1,
            max_tokens = 300,      
            google_api_key = self.API_KEY 
        )


        # Define actual final model 
        self.response_generation_llm = ChatGoogleGenerativeAI(
            model = "gemini-2.5-flash",
            temperature  = 0.3,
            max_tokens = 1024,
            google_api_key = self.API_KEY
        )

        # Other important managers 
        self.retreiver_manager = retreiver_manager

    # To refine the user query 
    def refine_query(self, query):

        # Define sample query which we pass into the llm 
        sample_query = f"""
            You rewrite search queries for a VillageTaste Foods business chatbot.
            
            The knowledge base contains:
            - food products
            - company services
            - employee policies
            - order information
            
            If the user query is generic or ambiguous,
            rewrite it by including the business/domain context.
            
            Improve the query for semantic retrieval.
            
            Keep the rewritten query concise.
            Do not use outside knowledge.
            Answer only from the provided context.
            
            User Query: {query}
            """

        # get refine query 
        refine_query = self.query_refine_llm.invoke(sample_query)

        # becoz at the end over refine_prompt exist inside the content key section 
        refine_query = refine_query.content.strip()
        
        return refine_query

    # Retreive refine query related documents 
    def retreive_documents(self, refine_query, top_k = 5):

        retreived_documents = self.retreiver_manager.retreive(refine_query, top_k = top_k)

        return retreived_documents
        
    # Retreived document related we generate the answer 
    def generate_answer(self, query, retreived_documents):

        # Check very imp edge case 
        if not retreived_documents:
            return """
                    I apologize, but your query does not appear to be related to VillageTaste Foods services or available information.
                    
                    Please ask questions related to:
                    - food products
                    - company services
                    - employee policies
                    - order information
                """ 

        else :
            
            # Exteact the real retreived document content 
            context = "\n\n".join([doc["document"] for doc in retreived_documents])

            # here we define over main prompt
            prompt = f"""
                    You are a professional AI assistant for VillageTaste Foods, an Indian village-based food business.
                    
                    Answer the user's question using ONLY the provided context information.
                    
                    Provide clear, helpful, and professional responses related to:
                    - food products
                    - services
                    - employee policies
                    - order information
                    
                    If the provided context does not contain enough information,
                    politely say that the information is unavailable.
                    Do NOT generate false information.
                    Keep the response natural and customer-friendly.
                    
                    Generate a concise but informative response in simple language.
                    
                    Focus on accuracy, professionalism, and grounded responses.
                    
                    Context: {context}
                    
                    User Question: {query}
                    """

            # Extract the response from the llm 
            response = self.response_generation_llm.invoke(prompt)

            return response.content

    # Join all pipline functions
    def chat(self, query, top_k = 5):

        refine_query = self.refine_query(query)

        retreived_documents = self.retreive_documents(refine_query, top_k = top_k)

        llm_response = self.generate_answer(query, retreived_documents)

        return {
            "llm_response": llm_response,
            "query": query,
            "refine_query": refine_query,
            "rertreived_documents": retreived_documents
        }

In [52]:
main_manager = RAGPipelineManager(retreiver_manager)

In [54]:
res = main_manager.chat("Give your Food list.")

# "What sweets do you sell?"
# "What services do you provide?"
# "What is quantum computing?"

In [ ]:
res

'Thank you for your interest in VillageTaste Foods!\n\nVillageTaste Foods specializes in preparing homemade traditional foods using authentic recipes passed down through generations. Our products are made with fresh ingredients, traditional spices, and village-style cooking methods, focusing on natural taste, hygiene, and quality.\n\nHowever, a specific list of our food products is not available in the information provided.'

# 🍲 VillageTaste Foods — Modular RAG Chatbot System

A domain-specific **Retrieval-Augmented Generation (RAG)** chatbot system designed for **VillageTaste Foods** to provide intelligent, context-aware, and service-specific responses using company knowledge documents.

This project demonstrates how modern AI systems can combine:

- Large Language Models (LLMs)
- Semantic Search
- Vector Databases
- Embedding Models
- Retrieval Pipelines

to build reliable business-focused AI assistants.

---

# 📌 What is RAG?

## Retrieval-Augmented Generation (RAG)

RAG is an AI architecture that combines:

1. **Retrieval**  
   → Fetch relevant information from external knowledge sources

2. **Generation**  
   → Use an LLM to generate accurate and context-aware responses

Instead of relying only on pretrained knowledge, RAG systems can access custom business data such as:

- PDFs
- DOCX files
- Text documents
- Excel sheets
- Company manuals
- FAQs
- Product catalogs

This helps reduce hallucination and improves response accuracy.

---

# ❓ Why This Project?

Traditional food businesses often have:

- Product catalogs
- Service details
- Festival offers
- Ordering instructions
- Ingredient information
- Customer FAQs

But customers usually need human support to access this information.

This project solves that problem by building an AI assistant that can:

- Understand user queries
- Retrieve relevant company information
- Generate intelligent responses
- Provide domain-specific assistance

for the **VillageTaste Foods** business ecosystem.

---

# 🧠 RAG Architecture Used

This project combines multiple RAG design concepts.

## ✅ Modular RAG Architecture

The entire pipeline is divided into reusable modules:

- ingestion
- chunking
- embeddings
- retrieval
- response generation

This makes the codebase scalable and maintainable.

---

## ✅ Domain-Specific RAG

The chatbot is focused only on:

- VillageTaste Foods
- Traditional food products
- Company services
- Recipes
- Customer support information

This improves retrieval precision and answer quality.

---

## ✅ Semantic Search RAG

The system uses embeddings and vector similarity search to retrieve semantically relevant documents instead of keyword matching.

---

## ✅ Query-Refinement RAG

Before retrieval, user queries are refined using an LLM to improve retrieval quality and context awareness.

---

# 🏗️ System Architecture Overview

# 1️⃣ Ingestion Pipeline

![Ingestion Pipeline](images/ingestion_pipline_overview.png)

The ingestion pipeline converts multiple document formats into vector-searchable knowledge chunks.

---

## 🔹 Supported Data Sources

The system supports:

- PDF files
- DOCX documents
- TXT files
- Excel spreadsheets

---

## 🔹 Document Conversion

Different loaders are used to convert raw documents into LangChain document objects.

### Examples

```python
PyMuPDFLoader()
TextLoader()
Docx2txtLoader()
UnstructuredExcelLoader()
```

Using LangChain documents helps standardize all data into a common format for downstream processing.

---

## 🔹 Smart Chunking Strategy

Chunking is one of the most important parts of a RAG pipeline.

Different document types require different chunking approaches.

### PDFs / DOCX

Used:
- paragraph-aware chunking
- separator-aware splitting

### TXT Files

Used:
- lightweight text splitting

### Excel Sheets

Used:
- row-wise chunking

This improves retrieval precision significantly.

---

## 🔹 Embedding Generation

The system converts document chunks into embeddings using:

```python
BAAI/bge-base-en-v1.5
```

### Why BGE?

Initially, sentence-transformer embeddings were tested, but retrieval quality was not sufficiently accurate for this domain.

BGE embeddings provided:

- better semantic understanding
- improved retrieval quality
- faster similarity matching
- more accurate context search

---

## 🔹 Vector Database Storage

All embeddings are stored in:

```python
ChromaDB
```

Stored data includes:

- chunk embeddings
- document content
- metadata
- unique identifiers

This enables efficient semantic retrieval.

---

# 2️⃣ Retrieval Pipeline

![Retrieval Pipeline](images/retrieval_pipline_overview.png)

The retrieval pipeline converts user queries into context-aware responses.

---

## 🔹 User Query

The user submits a natural language question.

### Example

```text
What homemade snacks are available?
```

---

## 🔹 Query Refinement Layer

Instead of directly embedding the raw query, the system first refines the query using an LLM.

This improves:

- retrieval accuracy
- domain awareness
- semantic matching

The LLM does NOT directly access the vector database.

It only improves the query structure for retrieval.

---

## 🔹 Semantic Retrieval

The refined query is converted into embeddings and searched inside ChromaDB.

The retriever returns:

```text
Top-K semantically relevant chunks
```

---

## 🔹 Final Response Generation

The final LLM receives:

- retrieved chunks
- user query

and generates a context-aware response.

---

# ⚙️ Important Engineering Decisions

# ✅ 1. Modular Programming Architecture

The project follows modular design principles.

Each pipeline component is separated into reusable classes and modules.

### Benefits

- maintainability
- scalability
- easier debugging
- reusable pipelines

---

# ✅ 2. Smart Chunking Strategy

Naive chunking often reduces retrieval quality.

This project uses document-specific chunking strategies.

| Document Type | Strategy |
|---|---|
| PDF | Paragraph-aware chunking |
| DOCX | Separator-aware chunking |
| TXT | Lightweight text splitting |
| Excel | Row-wise chunking |

---

# ✅ 3. Better Embedding Model Selection

Initial embedding models produced inconsistent retrieval quality.

The project was upgraded to:

```python
BAAI/bge-base-en-v1.5
```

which improved:

- semantic similarity
- retrieval precision
- contextual understanding

---

# ✅ 4. ChromaDB Similarity Metric Fix

By default, ChromaDB internally uses:

```text
L2 Distance
```

However, cosine similarity works better for semantic embeddings.

The system explicitly changes the similarity metric to:

```text
Cosine Similarity
```

to improve semantic search accuracy.

---

# ✅ 5. Query Refinement Before Retrieval

Directly embedding raw user queries can produce poor retrieval results.

The system improves retrieval quality by:

- refining the query
- injecting domain awareness
- improving semantic structure

before vector search.

---

# ✅ 6. Centralized RAGPipelineManager

A dedicated:

```python
RAGPipelineManager
```

class connects:

- ingestion pipeline
- retrieval pipeline
- embedding system
- LLM response generation

This creates a clean execution flow for the entire RAG system.

---

# 🛠️ Tech Stack

## AI / RAG Frameworks

- LangChain
- HuggingFace Transformers
- Sentence Transformers

---

## Embedding Model

```python
BAAI/bge-base-en-v1.5
```

---

## Vector Database

- ChromaDB

---

## LLM

- Gemini 2.5 Flash

---

## Data Processing

- Pandas
- PyMuPDF
- Docx2txt
- Unstructured

---

## Frontend

- Streamlit

---

## Programming Language

- Python

---

# 📂 Project Structure

```text
VILLAGETASTE-RAG-CHATBOT/
│
├── assets/
│   ├── banner.png
│   ├── logo.png
│   └── pot.png
│
├── data/
│   ├── csv_data/
│   ├── docs_data/
│   ├── pdfs_data/
│   ├── sheet_data/
│   ├── text_data/
│   │
│   └── vector_store/   -> Data Base Physical storage area 
│
├── ingestion/
│   ├── data_chunking_manager.py
│   ├── data_loader_manager.py
│   ├── embedding_manager.py
│   └── vector_store_manager.py
│
├── notebook/
│   └── Pipline_Implementation.ipynb
│
├── pipline/
│   └── rag_pipline_manager.py
│
├── retrieval/
│   └── retriever_manager.py
│
├── .env
│
├── app.py
├── main.py
│
├── README.md
│
└── requirement.txt
```

---

# 🚀 Installation

## 1️⃣ Clone Repository

```bash
git clone <your_repo_link>
```

---

## 2️⃣ Create Environment

```bash
conda create -n rag_env python=3.10
```

---

## 3️⃣ Activate Environment

```bash
conda activate rag_env
```

---

## 4️⃣ Install Dependencies

```bash
pip install -r requirements.txt
```

---

# ▶️ Run Streamlit App

```bash
streamlit run app.py
```

---

# 🔮 Future Improvements

Possible future upgrades:

- Flask / FastAPI backend
- Cloud deployment
- Hybrid search
- Reranking pipeline
- Conversational memory
- Multi-language support
- Advanced UI improvements
- Authentication system

---

# 📚 Learning Goals of This Project

This project was designed not only as a chatbot system but also as a beginner-friendly educational implementation of modern RAG architecture concepts.

The goal is to help learners understand:

- how RAG works internally
- why retrieval matters
- how embeddings improve search
- how vector databases work
- how modular AI systems are designed

---

# ❤️ Acknowledgment

Built as a learning-focused AI engineering project for exploring:

- Retrieval-Augmented Generation
- Semantic Search
- Vector Databases
- Modular AI Architectures
- Domain-Specific Chatbots

using real-world business use cases.